In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import re
import glob


In [ ]:
!pip install striprtf

In [ ]:
from striprtf.striprtf import rtf_to_text

In [ ]:
# file path
PATH = "/content/drive/MyDrive/Nachrichten"

# file list
rtf_files = glob.glob(f"{PATH}/*.RTF")

print(f"Found {len(rtf_files)} files")

Found 17 files


In [ ]:
all_data = []

for filename in rtf_files:

    print(f"Processing: {filename}")

    with open(filename, "r", encoding="utf-8", errors="ignore") as f:
        rtf = f.read()

    text = rtf_to_text(rtf)

    articles = text.split("End of Document")

    for article in articles:

        date_match = re.search(
            r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}',
            article
        )

        if not date_match:
            continue

        date = pd.to_datetime(date_match.group())

        body_match = re.search(
            r'Body(.*)',
            article,
            re.DOTALL
        )

        if body_match:

            body = body_match.group(1)

            body = re.sub(
                r'Load-Date:.*',
                '',
                body,
                flags=re.DOTALL
            )

            body = re.sub(r'\s+', ' ', body).strip()

            all_data.append({
                "date": date,
                "text": body,
                "source_file": filename.split("/")[-1]
            })


Processing: /content/drive/MyDrive/Nachrichten/501-750.RTF
Processing: /content/drive/MyDrive/Nachrichten/751-1000.RTF
Processing: /content/drive/MyDrive/Nachrichten/1001-1250.RTF
Processing: /content/drive/MyDrive/Nachrichten/1251-1580.RTF
Processing: /content/drive/MyDrive/Nachrichten/1-250.RTF
Processing: /content/drive/MyDrive/Nachrichten/251-500.RTF
Processing: /content/drive/MyDrive/Nachrichten/851-900.RTF
Processing: /content/drive/MyDrive/Nachrichten/901-950.RTF
Processing: /content/drive/MyDrive/Nachrichten/951-1000.RTF
Processing: /content/drive/MyDrive/Nachrichten/1001-1100.RTF
Processing: /content/drive/MyDrive/Nachrichten/1101-1200.RTF
Processing: /content/drive/MyDrive/Nachrichten/1201-1300.RTF
Processing: /content/drive/MyDrive/Nachrichten/1301-1400.RTF
Processing: /content/drive/MyDrive/Nachrichten/1401-1500.RTF
Processing: /content/drive/MyDrive/Nachrichten/1501-1600.RTF
Processing: /content/drive/MyDrive/Nachrichten/1601-1686.RTF
Processing: /content/drive/MyDrive/Nac

In [ ]:
df.to_parquet(
    "/content/drive/MyDrive/Nachrichten/news.parquet",
    compression="snappy",
    index=False
)

In [ ]:
df = pd.DataFrame(all_data)

print(f"Articles before deduplication: {len(df)}")

# REMOVE DUPLICATES

# use first 2000 characters of text to delete almost identical news

df["text_short"] = df["text"].str[:2000]

before = len(df)

df = df.drop_duplicates(
    subset=["date", "text_short"]
)

after = len(df)

df = df.drop(columns=["text_short"])

print(f"Articles after deduplication: {after}")
print(f"Removed duplicates: {before - after}")

# SORT BY DATE

df["date"] = pd.to_datetime(df["date"])

df = (
    df.sort_values("date")
      .reset_index(drop=True)
)

# MONTHLY AGGREGATION

df["month"] = df["date"].dt.to_period("M")

Articles before deduplication: 2616
Articles after deduplication: 2097
Removed duplicates: 519


In [ ]:
df

,date,text,source_file,month
0,2024-11-14,Prozent mehr betrug das Bruttoinlandsprodukt (...,1401-1500.RTF,2024-11
1,2024-11-14,Dass es so schnell keinen Bundeshaushalt für 2...,1201-1300.RTF,2024-11
2,2024-11-14,Siemens trotzt der schwachen Konjunktur und Sc...,1001-1100.RTF,2024-11
3,2024-11-14,VON QUIRIN HACKER Wäre die Podiumsdiskussion b...,1101-1200.RTF,2024-11
4,2024-11-14,Der Wahlsieg von Donald Trump hat die Finanzmä...,1501-1600.RTF,2024-11
...,...,...,...,...
2092,2025-05-28,"Annette Ludwig, Dorothee Schmidt-Ellmendorff u...",1001-1250.RTF,2025-05
2093,2025-05-28,Köln (dpa) Die deutsche Wirtschaft schrumpft i...,251-500.RTF,2025-05
2094,2025-06-05,Georg Winters Düsseldorf Kaum etwas dokumentie...,1-250.RTF,2025-06
2095,2025-06-05,Agentur für Arbeit hilft bei beruflicher Orien...,1251-1580.RTF,2025-06


In [ ]:
ifo_filename = "/content/drive/MyDrive/Nachrichten/gsk-e-202604.xlsx"

ifo = pd.read_excel(
    ifo_filename,
    sheet_name='ifo Business Climate',
    skiprows=9,
    usecols=[0,1,2,3]
)

ifo.columns = [
    'month',
    'business_climate',
    'business_situation',
    'business_expectations'
]

ifo['month'] = pd.to_datetime(
    ifo['month'].astype(str).str.strip(),
    format='%m/%Y'
)

ifo = ifo.dropna(subset=['business_climate'])

ifo['future_change'] = (
    ifo['business_climate'].shift(-1)
    - ifo['business_climate']
)

ifo['target'] = (
    ifo['future_change'] > 0
).astype(int)

ifo.head()

,month,business_climate,business_situation,business_expectations,future_change,target
0,2005-02-01,92.0,88.0,96.2,-1.9,0
1,2005-03-01,90.1,85.8,94.5,-0.2,0
2,2005-04-01,89.9,86.3,93.7,-0.6,0
3,2005-05-01,89.3,86.1,92.7,0.0,0
4,2005-06-01,89.3,85.6,93.2,1.8,1


In [ ]:
ifo = ifo.sort_values("month").reset_index(drop=True)

ifo = ifo.dropna(subset=["future_change"])

ifo["cutoff_date"] = (
    ifo["month"] - pd.DateOffset(months=1)
).apply(lambda x: x.replace(day=15))

ifo.head()

,month,business_climate,business_situation,business_expectations,future_change,target,cutoff_date
0,2005-02-01,92.0,88.0,96.2,-1.9,0,2005-01-15
1,2005-03-01,90.1,85.8,94.5,-0.2,0,2005-02-15
2,2005-04-01,89.9,86.3,93.7,-0.6,0,2005-03-15
3,2005-05-01,89.3,86.1,92.7,0.0,0,2005-04-15
4,2005-06-01,89.3,85.6,93.2,1.8,1,2005-05-15


In [ ]:
# -------------------------
# PREPARE IFO DATA
# -------------------------

ifo = ifo.sort_values("month").reset_index(drop=True)

ifo = ifo.dropna(subset=["future_change"])

# beginning of window: 15 days before the month
ifo["window_start"] = (
    ifo["month"] - pd.DateOffset(months=2)
).apply(lambda x: x.replace(day=15))

# end of window: 15 days after the month
ifo["window_end"] = (
    ifo["month"] - pd.DateOffset(months=1)
).apply(lambda x: x.replace(day=15))

# -------------------------
# PREPARE NEWS DATA
# -------------------------

df = df.sort_values("date").reset_index(drop=True)

df["target"] = np.nan
df["business_climate"] = np.nan
df["next_ifo_date"] = pd.NaT

# -------------------------
# MATCH NEWS TO IFO RELEASE
# -------------------------

for _, ifo_row in ifo.iterrows():

    mask = (
        (df["date"] >= ifo_row["window_start"]) &
        (df["date"] <= ifo_row["window_end"])
    )

    df.loc[mask, "target"] = ifo_row["target"]

    df.loc[mask, "business_climate"] = (
        ifo_row["business_climate"]
    )

    df.loc[mask, "next_ifo_date"] = (
        ifo_row["month"]
    )

# delete rows without a target
df = df.dropna(subset=["target"])

df["target"] = df["target"].astype(int)

print(df.shape)

df.head()

(2097, 7)


,date,text,source_file,month,target,business_climate,next_ifo_date
0,2024-11-14,Prozent mehr betrug das Bruttoinlandsprodukt (...,1401-1500.RTF,2024-11,1,85.0,2024-12-01
1,2024-11-14,Dass es so schnell keinen Bundeshaushalt für 2...,1201-1300.RTF,2024-11,1,85.0,2024-12-01
2,2024-11-14,Siemens trotzt der schwachen Konjunktur und Sc...,1001-1100.RTF,2024-11,1,85.0,2024-12-01
3,2024-11-14,VON QUIRIN HACKER Wäre die Podiumsdiskussion b...,1101-1200.RTF,2024-11,1,85.0,2024-12-01
4,2024-11-14,Der Wahlsieg von Donald Trump hat die Finanzmä...,1501-1600.RTF,2024-11,1,85.0,2024-12-01


In [ ]:
df.describe()

,date,target,business_climate,next_ifo_date
count,2097,2097.000000,2097.000000,2097
mean,2025-03-05 16:00:41.201716736,0.604673,87.286314,2025-04-07 06:43:46.609442304
min,2024-11-14 00:00:00,0.000000,85.000000,2024-12-01 00:00:00
25%,2024-12-09 00:00:00,0.000000,85.500000,2025-01-01 00:00:00
50%,2025-04-19 00:00:00,1.000000,88.300000,2025-06-01 00:00:00
75%,2025-05-02 00:00:00,1.000000,88.300000,2025-06-01 00:00:00
max,2025-06-05 00:00:00,1.000000,88.500000,2025-07-01 00:00:00
std,NaN,0.489037,1.296413,NaN


In [ ]:
print(int(df['target'].sum()))

1268


## Baseline model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score, f1_score

from tqdm.auto import tqdm
import pandas as pd

In [ ]:
# -------------------------
# TRAIN TEST SPLIT
# -------------------------

X = df["text"]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42,
    stratify=y
)

# -------------------------
# MODELS
# -------------------------

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Naive Bayes": MultinomialNB(),

    "Linear SVM": LinearSVC(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
# -------------------------
# TRAINING
# -------------------------

results = []

for model_name, clf in tqdm(
    models.items(),
    total=len(models),
    desc="Training models"
):

    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                max_features=10000,
                ngram_range=(1, 2),
                min_df=5
            )
        ),
        (
            "clf",
            clf
        )
    ])

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

# -------------------------
# RESULTS
# -------------------------

results_df = (
    pd.DataFrame(results)
      .sort_values("F1", ascending=False)
      .reset_index(drop=True)
)

print(results_df)

Training models:   0%|          | 0/4 [00:00<?, ?it/s]

                 Model  Accuracy        F1
0        Random Forest  0.854762  0.882466
1           Linear SVM  0.852381  0.876984
2  Logistic Regression  0.833333  0.866920
3          Naive Bayes  0.783333  0.821918
